# M3.1 Polarization and Bound Charge

Book pointer: chapter 5, sections 5.1 (dielectrics), 5.2 (bound charges).

## 1. Recall

1. What is a dipole moment, and how does a dielectric acquire a net dipole moment per volume when a field is applied?
2. Define the polarization $\mathbf{P}$. Units.
3. Write $\rho_b$ and $\sigma_b$ in terms of $\mathbf{P}$. Why does uniform polarization give only surface charge?
4. Argue in two sentences that bound charge is real charge, and that the total bound charge of a neutral body is zero.
5. A uniformly polarized sphere: where is the bound charge and what is the field inside?

## 2. Principle to algorithm

Polarization is a vector field on a grid like any other, so $\rho_b = -\nabla\cdot\mathbf{P}$ is the discrete divergence from module 1 with a minus sign. A surface charge is what the divergence produces when $\mathbf{P}$ steps to zero at a boundary: on a grid the step is spread over one or two cells, and integrating $\rho_b$ through those cells recovers $\sigma_b = \mathbf{P}\cdot\hat{\mathbf{n}}$.

Once the bound charges are known they are ordinary charges. The field of a polarized body is the superposition field of its bound charges, computed with `fields.point_charges_E` from module 1. That is the whole content of this notebook, and it is also the reason the result is a macroscopic field: it comes from the smooth charge densities, not from summing individual molecular dipoles. The last section shows why that distinction matters.

### Animation

Free charge builds up on the plates of a capacitor. In the right-hand panel a dielectric fills the gap: its molecules align with the field, bound charge appears on the two faces of the slab, and the field inside drops to $E_0/\epsilon_r$.

In [ ]:
from emlib import show
show.animation("m3_dielectrics", "PolarizationPanels")

## 3. Build

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from emlib import grids, fields, relax, dielectrics as di
from emlib.fields import EPS0, K

plt.rcParams["figure.figsize"] = (6, 5)

In [ ]:
# A non-uniform polarization and its volume bound charge.
x, y, z, G = grids.grid3d((-2, 2), (-2, 2), (-2, 2), 61)
h = grids.spacing(x, y, z)
X, Y, Z = grids.components(G)
P0 = 1e-6
Pol = np.zeros_like(G)
Pol[..., 0] = P0 * X * np.exp(-(X**2 + Y**2 + Z**2))         # P = P0 x exp(-r^2) x-hat

rho_b = di.bound_volume_charge(Pol, *h)
rho_b_exact = -P0 * (1 - 2 * X**2) * np.exp(-(X**2 + Y**2 + Z**2))

k = 30
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
im = axes[0].imshow(rho_b[:, :, k].T, origin="lower", extent=(-2, 2, -2, 2), cmap="coolwarm"); plt.colorbar(im, ax=axes[0], label="rho_b  [C/m^3]")
axes[0].quiver(X[::5, ::5, k], Y[::5, ::5, k], Pol[::5, ::5, k, 0], Pol[::5, ::5, k, 1], scale=8e-6); axes[0].set_title("rho_b = -div P in the plane z = 0")
axes[1].plot(x, rho_b[:, 30, k], label="numerical"); axes[1].plot(x, rho_b_exact[:, 30, k], "k--", label="exact")
axes[1].set_xlabel("x"); axes[1].set_ylabel("rho_b along y = z = 0"); axes[1].legend(); plt.show()
print(f"max |error| / max |rho_b| = {np.abs(rho_b - rho_b_exact)[2:-2, 2:-2, 2:-2].max() / np.abs(rho_b_exact).max():.2e}")
print(f"net bound charge / total |bound charge| = {abs(np.sum(rho_b)) / np.sum(np.abs(rho_b)):.1e}  (would vanish for a body entirely inside the box; here P is still {np.exp(-4):.3f} of its peak at the faces)")

### A uniformly polarized slab

$\mathbf{P} = P\,\hat{\mathbf{z}}$ for $|z| < t$, zero outside. There is no volume charge, but the discrete divergence sees the step at $z = \pm t$ and produces a spike one or two cells wide. Integrating the spike through the surface gives $\sigma_b = \pm P$.

In [ ]:
t = 1.0
slab = np.abs(Z) < t
Pslab = np.zeros_like(G); Pslab[..., 2] = P0 * slab
rho_slab = di.bound_volume_charge(Pslab, *h)

prof = rho_slab[30, 30, :]
plt.plot(z, prof, "o-", ms=3); plt.xlabel("z"); plt.ylabel("rho_b along the axis"); plt.title("surface charge appears as a spike at the faces"); plt.show()

top = (z > 0.5) & (z < 1.5); bottom = (z < -0.5) & (z > -1.5)
print(f"integral of rho_b through the top face:    {np.trapezoid(prof[top], dx=h[2]) / P0:+.4f} P   (sigma_b = P . n = +P)")
print(f"integral of rho_b through the bottom face: {np.trapezoid(prof[bottom], dx=h[2]) / P0:+.4f} P   (n = -z, so -P)")
print(f"from the formula: top {di.bound_surface_charge(np.array([0, 0, P0]), [0, 0, 1]) / P0:+.1f} P, bottom {di.bound_surface_charge(np.array([0, 0, P0]), [0, 0, -1]) / P0:+.1f} P")

### Exercise: bound charge of a polarized sphere

Replace the body of `sphere_bound_charges` so it returns patch centres, charges, and the patch area for a sphere of radius $R$ uniformly polarized along $z$, with $\sigma_b = P\cos\theta$. Use $N$ equal-area patches (the Fibonacci spiral in the cell is one way). The check requires the total charge to vanish and the interior field to be $-P/3\epsilon_0$.

In [ ]:
def fibonacci_sphere(R, N):
    k = np.arange(N) + 0.5
    theta = np.arccos(1 - 2 * k / N); phi = np.pi * (1 + 5**0.5) * k
    n = np.stack([np.sin(theta) * np.cos(phi), np.sin(theta) * np.sin(phi), np.cos(theta)], axis=-1)
    return R * n, n, 4 * np.pi * R**2 / N

def sphere_bound_charges(R, P, N):
    """Return (centres, charges) of N surface patches carrying sigma_b = P cos(theta)."""
    centres, normals, dA = fibonacci_sphere(R, N)
    sigma_b = di.bound_surface_charge(np.array([0.0, 0.0, P]), [0, 0, 1]) * normals[:, 2]   # replace with your own
    return centres, sigma_b * dA


R = 1.0
centres, qb = sphere_bound_charges(R, P0, 20000)
E_in = fields.point_charges_E(centres, qb, [[0, 0, 0], [0.3, 0.2, -0.4]])
print(f"total bound charge / sum|q| = {qb.sum() / np.abs(qb).sum():.1e}")
print(f"E_z at the centre and at an interior point: {E_in[:, 2]}   expected {-P0 / (3 * EPS0):.5e}")
assert np.allclose(E_in[:, 2], -P0 / (3 * EPS0), rtol=1e-2) and abs(qb.sum()) < 1e-9 * np.abs(qb).sum()

## 4. Verify

Along the $z$ axis the field of the polarized sphere has two closed forms. Inside, $E_z = -P/3\epsilon_0$. Outside, the sphere is a pure dipole with $p = \frac{4}{3}\pi R^3 P$, so $E_z(z) = \dfrac{2p}{4\pi\epsilon_0 z^3}$.

In [ ]:
zz = np.concatenate([np.linspace(-3, -1.05, 60), np.linspace(-0.95, 0.95, 60), np.linspace(1.05, 3, 60)])
Ez = fields.point_charges_E(centres, qb, np.stack([0 * zz, 0 * zz, zz], axis=-1))[:, 2]
p = 4 / 3 * np.pi * R**3 * P0
exact = np.where(np.abs(zz) < R, -P0 / (3 * EPS0), 2 * p / (4 * np.pi * EPS0 * np.abs(zz) ** 3))
plt.plot(zz, exact, "k-", lw=3, alpha=0.3, label="closed forms"); plt.plot(zz, Ez, "C1.", ms=4, label="bound-charge patches")
plt.axvline(-R, color="grey", ls=":"); plt.axvline(R, color="grey", ls=":")
plt.xlabel("z / R"); plt.ylabel("E_z  [V/m]"); plt.legend(); plt.show()
rel = np.abs(Ez / exact - 1)
print(f"max relative error inside {rel[np.abs(zz) < R].max():.2e}, outside {rel[np.abs(zz) > R].max():.2e}")
assert rel.max() < 2e-2

## 5. Explore and rebuild

Explore. Fill the same sphere with a cubic lattice of point dipoles of moment $P h^3$ each, using `di.dipole_sources`, and evaluate the field at a lattice site (excluding the dipole that sits there) and at a few points between sites. Compare with $-P/3\epsilon_0$. The site value is close to zero and the off-site values swing both ways. Explain why the macroscopic field is the average of this microscopic field over a cell and why the bound-charge route gives the macroscopic field directly. Section 5.2 is the place to check your reasoning.

Rebuild. Close this notebook and write `bound_volume_charge`, `bound_surface_charge`, and the sphere-patch construction from scratch.